# Baseline Models for Yelp Review Analysis

This notebook:
1. Loads cleaned Yelp review data from 'cleaned-data/'.
2. Applies TF-IDF feature extraction with n-grams.
3. Trains basic ML models: Logistic Regression, Naive Bayes, SVM, Random Forest, and Ordinal Regression.
4. Reports standard performance metrics for both sentiment and numerical rating prediction.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from mord import LogisticAT
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import glob
import os

In [2]:
folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))

dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state'] = os.path.splitext(os.path.basename(file))[0] 
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
df = data.dropna()
df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

print(f"Loaded {len(df)} reviews from {len(dfs)} states.")

Loaded 5222860 reviews from 20 states.


/var/folders/cl/5mfvhcls2nv2h9gy15m04b640000gn/T/ipykernel_33249/3678554632.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))


In [3]:
def create_tfidf_features(X_train, X_test, max_features=10000, ngram_range=(1, 2)):

    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    
    return X_train_tfidf, X_test_tfidf, vectorizer


In [4]:
def train_baseline_models(X_train, y_train, X_test, y_test):
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Naive Bayes': MultinomialNB(),
        'SVM': SVC(kernel='linear', max_iter=2000),
        'Random Forest': RandomForestClassifier(n_estimators=200, max_features=100, max_depth=25),
        'Ordinal Logistic Regression': LogisticAT(max_iter=2000)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        
        # Per-class metrics
        report = classification_report(y_test, y_pred, output_dict=True)
        
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'predictions': y_pred,
            'classification_report': report
        }
        
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro-F1: {macro_f1:.4f}")
    
    return results

In [5]:
def main(data, n):
    
    print("Prepping data for modeling...")
    print()

    # Determine per-class sample size
    if n is None:
        samples_per_class = data['stars'].value_counts().min()
    else:
        samples_per_class = min(n // data['stars'].nunique(), data['stars'].value_counts().min())

    # Balanced sampling
    data_sample = (
        data.groupby('stars', group_keys=False)
            .apply(lambda x: x.sample(n=samples_per_class))
            .reset_index(drop=True)
    )

    print("Class distribution in subsample:")
    print(data_sample['stars'].value_counts())
    print()

    X = data_sample['clean_text']
    y_binary = data_sample['sentiment'].astype(int)
    y_rrp = data_sample['stars'].astype(int)
        
    # data = data.sample(n)
    # X = data['clean_text']
    # y_binary = data['sentiment'].astype(int)
    # y_rrp = data['stars'].astype(int)

    Xb_train, Xb_test, yb_train, yb_test = train_test_split(X, y_binary, test_size=0.1, random_state=42)
    Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_rrp, test_size=0.1, random_state=42)

    print(f"Training set: {len(Xb_train)} reviews")
    print(f"Test set: {len(Xb_test)} reviews")

    # Create features
    print()
    print("Creating n-gram/TF-IDF features...")
    Xb_train_tfidf, Xb_test_tfidf, _ = create_tfidf_features(Xb_train, Xb_test, max_features=10000, ngram_range=(1, 2))
    Xr_train_tfidf, Xr_test_tfidf, _ = create_tfidf_features(Xr_train, Xr_test, max_features=10000, ngram_range=(1, 2))

    # Train baseline models (binary)
    print("\n" + "="*60)
    print("BASELINE MODELS (binary)")
    print("="*60)
    results_binary = train_baseline_models(Xb_train_tfidf, yb_train, Xb_test_tfidf, yb_test)
    
    # Train baseline models (1-5 RRP)
    print("\n" + "="*60)
    print("BASELINE MODELS (1-5 RRP)")
    print("="*60)
    results_rrp = train_baseline_models(Xr_train_tfidf, yr_train, Xr_test_tfidf, yr_test)
    
    # Compare all results
    all_results = {}
    for name, metrics in results_rrp.items():
        all_results[f"{name} (RRP)"] = metrics
    for name, metrics in results_binary.items():
        all_results[f"{name} (BINARY)"] = metrics
    
    all_results

In [6]:
main(data=df, n=500000)

Prepping data for modeling...

Class distribution in subsample:
1.0    100000
2.0    100000
3.0    100000
4.0    100000
5.0    100000
Name: stars, dtype: int64

Training set: 450000 reviews
Test set: 50000 reviews

Creating n-gram/TF-IDF features...

BASELINE MODELS (binary)

Training Logistic Regression...
Accuracy: 0.8103
Macro-F1: 0.7539

Training Naive Bayes...
Accuracy: 0.7741
Macro-F1: 0.6969

Training SVM...


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Accuracy: 0.5637
Macro-F1: 0.5206

Training Random Forest...
Accuracy: 0.7244
Macro-F1: 0.5386

Training Ordinal Logistic Regression...
Accuracy: 0.7803
Macro-F1: 0.7256

BASELINE MODELS (1-5 RRP)

Training Logistic Regression...
Accuracy: 0.6212
Macro-F1: 0.6190

Training Naive Bayes...
Accuracy: 0.5851
Macro-F1: 0.5826

Training SVM...


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Accuracy: 0.4149
Macro-F1: 0.4125

Training Random Forest...
Accuracy: 0.5339
Macro-F1: 0.5172

Training Ordinal Logistic Regression...
Accuracy: 0.5391
Macro-F1: 0.5429
